# Kaggriculture | Crop Timing and Labor Budget

This is a rules-derived planning notebook for crop timing, shed pressure and farm-hand cost curves. It is not an
agent, not a simulator, not a submission, and not a leaderboard claim.

Source: Kaggle official `Kaggle/kaggle-environments` Kaggriculture environment at commit
`bbda347572cf5134e56f0eb49e8058e2560f9844`, Apache-2.0. Only compact parameter tables are embedded here.

**Checked results:** daily-watered, unfertilized Melon reaches 6 units at age 10, not age 12.
Twelve earliest hires cost 376 per day and give at most 298 unit-action slots including the farmer.
A 25-tile Melon harvest is 150 units versus a 100-unit shed: sales and deposit timing matter.

[Pinned official source](https://github.com/Kaggle/kaggle-environments/blob/bbda347572cf5134e56f0eb49e8058e2560f9844/kaggle_environments/envs/kaggriculture/kaggriculture.py),
[competition rules](https://www.kaggle.com/competitions/kaggriculture/rules).
Parameter excerpts are attributed to Kaggle and upstream contributors; calculations and charts here are new,
prepared with AI assistance. The upstream license is included below. No third-party agent is copied.

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('ggplot')
working = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('work/multi_competition/local_runs/kaggriculture')
working.mkdir(parents=True, exist_ok=True)

In [ ]:
CROPS = {
    'WHEAT':      {'seed': 10, 'first_yield_day': 2, 'max_yield_day': 4,  'interval': 0, 'max_yield': 6, 'ongoing': False, 'base_price': 25},
    'CARROT':     {'seed': 20, 'first_yield_day': 2, 'max_yield_day': 3,  'interval': 0, 'max_yield': 4, 'ongoing': False, 'base_price': 35},
    'TOMATO':     {'seed': 50, 'first_yield_day': 8, 'max_yield_day': 8,  'interval': 1, 'max_yield': 4, 'ongoing': True,  'base_price': 60},
    'STRAWBERRY': {'seed': 100,'first_yield_day': 10,'max_yield_day': 10, 'interval': 2, 'max_yield': 4, 'ongoing': True,  'base_price': 120},
    'MELON':      {'seed': 80, 'first_yield_day': 10,'max_yield_day': 12, 'interval': 0, 'max_yield': 6, 'ongoing': False, 'base_price': 250},
}
DEFAULTS = {'episodeSteps': 720, 'turnsPerDay': 24, 'shedCapacity': 100, 'maxMarketOrdersPerTurn': 10, 'farmHandCostMult': 1}
MARKET = {
    'WHEAT': {'base': 25, 'T': 400, 'below_func': 'sqrt', 'below_target': 0.80, 'above_func': 'log', 'above_target': 0.20},
    'CARROT': {'base': 35, 'T': 450, 'below_func': 'hinge', 'below_target': 1.00, 'above_func': 'sqrt', 'above_target': 0.70},
    'TOMATO': {'base': 60, 'T': 200, 'below_func': 'hinge', 'below_target': 0.40, 'above_func': 'sqrt', 'above_target': 0.60},
    'STRAWBERRY': {'base': 120, 'T': 100, 'below_func': 'sqrt', 'below_target': 0.70, 'above_func': 'linear', 'above_target': 1.60},
    'MELON': {'base': 250, 'T': 300, 'below_func': 'log', 'below_target': 0.20, 'above_func': 'sq', 'above_target': 3.60},
}

In [ ]:
def unfertilized_daily_yield(crop, age):
    c = CROPS[crop]
    if c['ongoing']:
        if age < c['first_yield_day']:
            return 0
        if (age - c['first_yield_day']) % c['interval'] != 0:
            return 0
        production_index = (age - c['first_yield_day']) // c['interval'] + 1
        return 1 if production_index <= c['max_yield'] else 0
    window_start = (c['max_yield_day'] + 1) // 2
    peak_age = min(c['max_yield_day'], window_start + c['max_yield'] - 2)
    units = min(c['max_yield'], 1 + peak_age - window_start + 1)
    return units if age == peak_age else 0

crop_rows = []
for crop, c in CROPS.items():
    cumulative = 0
    last_positive = None
    peak_age = None
    peak_units = -1
    for age in range(0, 20):
        y = unfertilized_daily_yield(crop, age)
        cumulative += y
        if y > 0:
            last_positive = age
        if cumulative > peak_units:
            peak_units = cumulative
            peak_age = age
        crop_rows.append({'crop': crop, 'age': age, 'yield_units_that_day': y, 'cumulative_units': cumulative})

crop_df = pd.DataFrame(crop_rows)
crop_summary = crop_df.groupby('crop').agg(
    total_unfertilized_units=('yield_units_that_day', 'sum'),
    last_effective_age=('age', lambda s: int(s[crop_df.loc[s.index, 'yield_units_that_day'] > 0].max())),
).reset_index()
crop_summary['seed_cost'] = crop_summary['crop'].map(lambda x: CROPS[x]['seed'])
crop_summary['base_price'] = crop_summary['crop'].map(lambda x: CROPS[x]['base_price'])
crop_summary['gross_at_base'] = crop_summary['total_unfertilized_units'] * crop_summary['base_price']
crop_summary['net_seed_only'] = crop_summary['gross_at_base'] - crop_summary['seed_cost']
crop_summary['occupied_days_denominator'] = crop_summary['last_effective_age'] + 1
crop_summary['net_per_occupied_day'] = crop_summary['net_seed_only'] / crop_summary['occupied_days_denominator']
expected = {'WHEAT': (4, 4), 'CARROT': (3, 3), 'TOMATO': (4, 11), 'STRAWBERRY': (4, 16), 'MELON': (6, 10)}
for row in crop_summary.itertuples():
    assert (row.total_unfertilized_units, row.last_effective_age) == expected[row.crop]
crop_summary['field_units'] = 25 * crop_summary['total_unfertilized_units']
crop_summary.sort_values('net_per_occupied_day', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for crop in CROPS:
    sub = crop_df[crop_df['crop'] == crop]
    axes[0].plot(sub['age'], sub['cumulative_units'], marker='o', label=crop)
axes[0].set_title('Unfertilized cumulative crop units')
axes[0].set_xlabel('Age in days')
axes[0].set_ylabel('Units')
axes[0].legend(fontsize=8)

crop_summary.sort_values('net_per_occupied_day').plot.barh(x='crop', y='net_per_occupied_day', ax=axes[1], color='#3a7f63', legend=False)
axes[1].set_title('Seed-only net per occupied day')
axes[1].set_xlabel('Money / tile / day at base price')

crop_summary.plot.bar(x='crop', y='field_units', ax=axes[2], color='#9b6538', legend=False)
axes[2].axhline(DEFAULTS['shedCapacity'], color='black', linewidth=1, linestyle='--')
axes[2].set_title('25-tile lifetime yield vs shed capacity')
axes[2].set_ylabel('Product units (not all simultaneous)')
fig.tight_layout()
fig.savefig(working / 'kaggriculture_crop_timing.png', dpi=160)
plt.show()

In [ ]:
def fib_sequence(n):
    seq = []
    a, b = 1, 1
    for _ in range(n):
        seq.append(a)
        a, b = b, a + b
    return seq

hires = list(range(0, 21))
fibs = fib_sequence(max(hires) + 1)
labor_rows = []
for n in hires:
    total_cost = sum(fibs[:n]) * DEFAULTS['farmHandCostMult']
    optimistic_hand_actions = sum(max(0, DEFAULTS['turnsPerDay'] - 1 - i // DEFAULTS['maxMarketOrdersPerTurn']) for i in range(n))
    labor_rows.append({
        'hires': n,
        'daily_hire_cost': total_cost,
        'optimistic_total_actions': DEFAULTS['turnsPerDay'] + optimistic_hand_actions,
        'cost_per_extra_action': total_cost / optimistic_hand_actions if optimistic_hand_actions else np.nan,
    })
labor_df = pd.DataFrame(labor_rows)
assert labor_df.loc[12, 'daily_hire_cost'] == 376
assert labor_df.loc[12, 'optimistic_total_actions'] == 298

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labor_df.plot(x='hires', y='daily_hire_cost', marker='o', ax=axes[0], color='#304f8c', legend=False)
axes[0].set_title('Fibonacci daily hire cost')
axes[0].set_ylabel('Money')
labor_df.plot(x='hires', y='optimistic_total_actions', marker='o', ax=axes[1], color='#8f3f55', legend=False)
axes[1].set_title('Optimistic daily action ceiling')
axes[1].set_ylabel('Actions including farmer')
fig.tight_layout()
fig.savefig(working / 'kaggriculture_labor_budget.png', dpi=160)
plt.show()

labor_df.head(13)

In [ ]:
market_df = pd.DataFrame(MARKET).T.reset_index().rename(columns={'index': 'product'})
crop_summary.to_csv(working / 'kaggriculture_crop_summary.csv', index=False)
labor_df.to_csv(working / 'kaggriculture_labor_budget.csv', index=False)
market_df.to_csv(working / 'kaggriculture_market_params.csv', index=False)
summary = {
    'competition': 'kaggriculture',
    'source_commit': 'bbda347572cf5134e56f0eb49e8058e2560f9844',
    'apache_2_0_parameter_excerpt': True,
    'defaults': DEFAULTS,
    'crop_order_by_net_per_occupied_day': crop_summary.sort_values('net_per_occupied_day', ascending=False)['crop'].tolist(),
    'twelve_hires_daily_cost': int(labor_df.loc[labor_df['hires'] == 12, 'daily_hire_cost'].iloc[0]),
    'twelve_hires_optimistic_actions': int(labor_df.loc[labor_df['hires'] == 12, 'optimistic_total_actions'].iloc[0]),
    'submission_created': False,
    'leaderboard_score': None,
}
(working / 'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## Practical use

The immediate ablation is not “copy a full strategy.” It is to test schedule constraints:
how many farm hands are actually worth buying before route congestion and shed capacity erase the theoretical
action advantage, and whether wheat is better treated as direct revenue or as feed/logistics infrastructure.

Assumptions: daily watering, no fertilizer, one-time crops harvested once at peak, ongoing crops collected
without decay. Seed-only net excludes labor, movement, land, weeds and dynamic market prices; it is not profit.
Lifetime field yield is not a simultaneous shed deposit. Hire capacity assumes earliest affordable hiring,
all market slots reserved for HIRE, and no congestion: new hands miss the hiring turn's unit action.
Independent agent experiments are required before using any ranking as a strategy recommendation.

## Upstream license

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```